<a href="https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/Copy_of_w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The purpose of this queue is to help a content team decide which pages are worth reviewing first. The model score is treated as a directional signal, not as proof that a page needs a refresh.

Pages with stronger scores are placed higher in the review queue. Each page also gets a simple reason code so that the recommendation is easier for a person to understand.

The main action is **REVIEW_REFRESH**, meaning the page should be reviewed for possible content updates. The reason codes describe the signals behind the recommendation rather than claiming that a refresh will definitely improve performance.

I will prioritize pages showing a stronger decline signal, especially when they also have meaningful impressions or appear to have become stale. The final decision should still be made by a person after checking the actual content and search context.

In [ ]:
!git clone https://github.com/Khadija-Azam05/ML-Projects.git

Cloning into 'ML-Projects'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (196/196), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 196 (delta 84), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (196/196), 1.90 MiB | 12.07 MiB/s, done.
Resolving deltas: 100% (84/84), done.


In [ ]:
import pandas as pd
import numpy as np

# Load the FlyRank starter dataset
df = pd.read_csv("ML-Projects/data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

# --------------------------------------------------
# 1. Create a lightweight review score
# --------------------------------------------------

queue = df.copy()

# Normalize impressions
queue["impression_score"] = (
    queue["impressions_90d"] / queue["impressions_90d"].max()
).clip(0, 1)

# Higher position number = weaker position,
# so convert it into a review-priority signal.
queue["position_score"] = (
    1 - (queue["avg_position"] / queue["avg_position"].max())
).clip(0, 1)

# Normalize content age
queue["stale_score"] = (
    queue["days_since_last_update"] /
    queue["days_since_last_update"].max()
).clip(0, 1)

# Lightweight prioritization score
queue["review_score"] = (
    0.5 * queue["impression_score"]
    + 0.3 * queue["position_score"]
    + 0.2 * queue["stale_score"]
)

# --------------------------------------------------
# 2. Assign human-readable reason codes
# --------------------------------------------------

queue["reason_code"] = "REVIEW_FIRST"

# Pages not updated for 90+ days
queue.loc[
    queue["days_since_last_update"] >= 90,
    "reason_code"
] = "STALE_CONTENT"

# High-visibility pages that are not already classified as stale
queue.loc[
    (queue["impressions_90d"] >= queue["impressions_90d"].quantile(0.75))
    & (queue["days_since_last_update"] < 90),
    "reason_code"
] = "HIGH_VISIBILITY_REVIEW"

# --------------------------------------------------
# 3. Assign action
# --------------------------------------------------

queue["action"] = "REVIEW_REFRESH"

# --------------------------------------------------
# 4. Rank the queue
# --------------------------------------------------

queue = queue.sort_values(
    "review_score",
    ascending=False
).reset_index(drop=True)

# --------------------------------------------------
# 5. Display the top 10
# --------------------------------------------------

top10 = queue[
    [
        "review_score",
        "reason_code",
        "action",
        "impressions_90d",
        "clicks_90d",
        "avg_position",
        "days_since_last_update"
    ]
].head(10)

print("\nTop 10 recommended actions:")
display(top10)

Dataset shape: (30000, 44)

Top 10 recommended actions:


,review_score,reason_code,action,impressions_90d,clicks_90d,avg_position,days_since_last_update
0,0.850621,STALE_CONTENT,REVIEW_REFRESH,517715,741,4.2,104
1,0.804599,HIGH_VISIBILITY_REVIEW,REVIEW_REFRESH,517109,1270,5.4,22
2,0.799489,HIGH_VISIBILITY_REVIEW,REVIEW_REFRESH,509252,785,2.5,20
3,0.779250,HIGH_VISIBILITY_REVIEW,REVIEW_REFRESH,497727,487,22.2,48
4,0.755164,HIGH_VISIBILITY_REVIEW,REVIEW_REFRESH,463103,1889,2.3,20
5,0.749862,STALE_CONTENT,REVIEW_REFRESH,443434,910,27.9,104
6,0.708838,HIGH_VISIBILITY_REVIEW,REVIEW_REFRESH,416180,944,4.0,22
7,0.686133,STALE_CONTENT,REVIEW_REFRESH,347399,1854,4.2,104
8,0.651927,STALE_CONTENT,REVIEW_REFRESH,309192,2689,2.0,104
9,0.648213,STALE_CONTENT,REVIEW_REFRESH,309910,492,5.6,104


## 1. Ranked actions + reason codes

The purpose of this queue is to help a content team decide which pages should be reviewed first. I use a simple review score based on visibility, average position, and time since the last update.

The queue uses three reason codes: **STALE_CONTENT**, **HIGH_VISIBILITY_REVIEW**, and **REVIEW_FIRST**. These codes explain why a page was prioritized without claiming that the page definitely needs a refresh.

The highest-ranked pages are candidates for human review. A high score is a directional decision-support signal, not proof that refreshing a page will improve its performance.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is intended for content teams that need a practical way to prioritize pages for human review. It can help identify pages that are more visible, appear weaker in search position, or have gone a long time without an update.

The recommendations are directional and should be used as decision-support rather than automatic decisions. A high review score does not prove that a page is declining, that its content is outdated, or that a refresh will improve performance.

The playbook should not be used to automatically rewrite, delete, publish, or merge content. Before taking action, a person should review the page, its search intent, current content quality, and relevant performance context.

The model was also more conservative under the grouped validation used in ML-09, where NDCG@100 decreased from the Week-5 random-split result of 0.926767 to 0.769499. This difference is a reminder that the ranking should be treated as directional rather than guaranteed performance.

In [ ]:
print("Intended use: prioritize pages for human content review.")
print("Decision type: decision-support, not automatic publishing.")
print("Random-split NDCG@100:", 0.926767)
print("Grouped-split NDCG@100:", 0.769499)
print("Validation gap:", round(0.926767 - 0.769499, 6))

Intended use: prioritize pages for human content review.
Decision type: decision-support, not automatic publishing.
Random-split NDCG@100: 0.926767
Grouped-split NDCG@100: 0.769499
Validation gap: 0.157268


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## My Thoughts

The ranked queue is a starting point for human review, not an automatic action system. Before acting on a recommendation, a content team should check the page itself, its search intent, the quality and relevance of the content, recent performance, and whether the page is still an appropriate candidate for improvement.

A reviewer should also check whether the reason code makes sense for the specific page. For example, a **STALE_CONTENT** flag only indicates that the page has not been updated recently; it does not by itself mean that the content is outdated or incorrect.

The following decisions should not be automated by this playbook:

- Publishing or rewriting content without human review.
- Deleting or pruning a page solely because it receives a low score.
- Merging pages automatically.
- Claiming that a refresh will cause a performance improvement.
- Treating the ranking as proof of how a search engine's algorithm works.
- Making high-impact content decisions without checking the underlying page and search context.

The intended role of the system is to reduce the amount of manual searching needed to find review candidates while keeping the final decision with a human.

In [ ]:
print("Human review required: YES")
print("Automatic publishing: NO")
print("Automatic deletion/pruning: NO")
print("Automatic rewriting: NO")
print("Causal claims about refresh impact: NO")
print("Primary use: decision-support for content review")

Human review required: YES
Automatic publishing: NO
Automatic deletion/pruning: NO
Automatic rewriting: NO
Causal claims about refresh impact: NO
Primary use: decision-support for content review


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. My Thoughts

The recommendations should be monitored because search performance and content conditions can change over time. A queue that was useful on the current data may become less useful if the underlying patterns change.

I would review the system when the distribution of important signals changes noticeably, when the ranked recommendations stop matching useful human review candidates, or when new validation results show a meaningful drop in ranking performance.

A retraining or re-evaluation trigger would include:

- A noticeable change in the distribution of impressions, clicks, position, or update age.
- A decline in ranking performance on a new validation period.
- Repeated false or low-value recommendations during human review.
- Changes in the available data or the way key fields are measured.
- A change in the content team's use case that makes the current ranking logic unsuitable.

The model should be re-evaluated before being trusted on substantially different data. Monitoring is intended to detect when the current recommendations may no longer be reliable, rather than assuming that the current ranking will remain valid indefinitely.

In [ ]:
print("Monitoring checks:")

print("- Signal distributions: monitor")
print("- Ranking performance: re-evaluate")
print("- Human-review quality: monitor")
print("- Data/measurement changes: check")
print("- Major use-case changes: re-evaluate")

print("\nRetrain/re-evaluate when current recommendations no longer match")
print("observed validation results or useful human review outcomes.")

Monitoring checks:
- Signal distributions: monitor
- Ranking performance: re-evaluate
- Human-review quality: monitor
- Data/measurement changes: check
- Major use-case changes: re-evaluate

Retrain/re-evaluate when current recommendations no longer match
observed validation results or useful human review outcomes.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## My Thoughts

The ranked action queue will be exported so that the research paper can use the same results produced by this notebook.

The export contains the review score, reason code, recommended action, and the main signals used to explain the ranking. It is intended as a reproducible output of the notebook rather than a production data feed.

The queue is regenerated when the notebook is run, so the paper can trace its recommendations back to this analysis.

In [ ]:
import os

# Create the output directory if it does not already exist
os.makedirs("ML-Projects/work/outputs", exist_ok=True)

# Select the fields needed for the action queue
export_columns = [
    "review_score",
    "reason_code",
    "action",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "days_since_last_update"
]

paper_queue = queue[export_columns].copy()

# Save the ranked queue
output_path = "ML-Projects/work/outputs/action_playbook_queue.csv"

paper_queue.to_csv(output_path, index=False)

print("Queue exported successfully.")
print("Rows exported:", len(paper_queue))
print("Output path:", output_path)

print("\nTop 10 exported rows:")
display(paper_queue.head(10))

Queue exported successfully.
Rows exported: 30000
Output path: ML-Projects/work/outputs/action_playbook_queue.csv

Top 10 exported rows:


,review_score,reason_code,action,impressions_90d,clicks_90d,avg_position,days_since_last_update
0,0.850621,STALE_CONTENT,REVIEW_REFRESH,517715,741,4.2,104
1,0.804599,HIGH_VISIBILITY_REVIEW,REVIEW_REFRESH,517109,1270,5.4,22
2,0.799489,HIGH_VISIBILITY_REVIEW,REVIEW_REFRESH,509252,785,2.5,20
3,0.779250,HIGH_VISIBILITY_REVIEW,REVIEW_REFRESH,497727,487,22.2,48
4,0.755164,HIGH_VISIBILITY_REVIEW,REVIEW_REFRESH,463103,1889,2.3,20
5,0.749862,STALE_CONTENT,REVIEW_REFRESH,443434,910,27.9,104
6,0.708838,HIGH_VISIBILITY_REVIEW,REVIEW_REFRESH,416180,944,4.0,22
7,0.686133,STALE_CONTENT,REVIEW_REFRESH,347399,1854,4.2,104
8,0.651927,STALE_CONTENT,REVIEW_REFRESH,309192,2689,2.0,104
9,0.648213,STALE_CONTENT,REVIEW_REFRESH,309910,492,5.6,104


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.